# Tema: INSERT, UPDATE y DELETE

## Objetivos
Aplicar cambios selectivos y comprobar su alcance.

## Conceptos importantes para el examen
Predicados; transacciones; idempotencia; APPEND frente a sustitución.

**Dificultad:** Básico · **Tiempo estimado:** 50 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_05_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Insertar

In [ ]:
%sql
CREATE TABLE dml_demo USING DELTA AS SELECT * FROM employees;
INSERT INTO dml_demo SELECT 50, name, department, salary, active, created_at, updated_at FROM employees WHERE employee_id = 1;

### 2. Actualizar con condición

In [ ]:
%sql
UPDATE dml_demo SET salary = 50000, updated_at = TIMESTAMP '2026-02-01' WHERE employee_id = 50;
SELECT * FROM dml_demo WHERE employee_id = 50;

### 3. Eliminar e inspeccionar

In [ ]:
%sql
DELETE FROM dml_demo WHERE employee_id = 50;
DESCRIBE HISTORY dml_demo;

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea dml_practice desde employees y añade un empleado 99.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Sube un 5% los salarios de Data activos; consulta primero las filas afectadas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Elimina únicamente los inactivos de Sales.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Asigna salario fijo 52.000 a 99, ejecuta dos veces y comprueba que el resultado no cambia.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Muestra las métricas de las últimas operaciones y comprueba que no hay duplicados de ID.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** CTAS y una proyección explícita.

**Pista 2:** Repetir una subida porcentual no es idempotente.

**Pista 3:** Combina ambos predicados.

**Pista 4:** Un valor absoluto permite idempotencia de datos.

**Pista 5:** DESCRIBE HISTORY devuelve operationMetrics.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TABLE dml_practice USING DELTA AS SELECT * FROM employees;
INSERT INTO dml_practice SELECT 99, 'Nueva persona', 'Data', 48000, true, TIMESTAMP '2026-02-01', TIMESTAMP '2026-02-01';

### Solución 2

In [ ]:
%sql
SELECT employee_id, salary FROM dml_practice WHERE department = 'Data' AND active;
UPDATE dml_practice SET salary = CAST(ROUND(salary * 1.05) AS INT) WHERE department = 'Data' AND active;

### Solución 3

In [ ]:
%sql
DELETE FROM dml_practice WHERE department = 'Sales' AND NOT active;
SELECT * FROM dml_practice WHERE department = 'Sales';

### Solución 4

In [ ]:
%sql
UPDATE dml_practice SET salary = 52000 WHERE employee_id = 99;
UPDATE dml_practice SET salary = 52000 WHERE employee_id = 99;
SELECT * FROM dml_practice WHERE employee_id = 99;

### Solución 5

In [ ]:
display(spark.sql("DESCRIBE HISTORY dml_practice").select("version", "operation", "operationMetrics"))
assert spark.table("dml_practice").groupBy("employee_id").count().filter("count > 1").count() == 0

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué ocurre al repetir salary = salary * 1.05?

A. Es siempre idempotente

B. Se vuelve a incrementar

C. No se ejecuta

D. Solo cambia el esquema

### Pregunta 2
¿Cómo anticipas el alcance de DELETE?

A. VACUUM

B. DROP SCHEMA

C. SELECT con el mismo WHERE

D. OPTIMIZE

### Pregunta 3
¿Qué registro ayuda a revisar filas modificadas?

A. operationMetrics del historial

B. Nombre del notebook

C. Tamaño del widget

D. Catálogo por defecto

### Respuestas y explicación
**1. B** — El cálculo usa el salario ya aumentado.

**2. C** — El predicado muestra las filas candidatas.

**3. A** — Las métricas describen la operación confirmada.

## PARTE 6 - RETO FINAL
Procesa una baja, una contratación y una revisión salarial fija. Demuestra qué pasos se pueden repetir y convierte la inserción en una carga sin duplicados.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
